In [1]:
import torch
import numpy as np
import torch.nn as nn

In [2]:
# 忽略 bias，把 w_o单独列出来
# k，q，v  的计算合并成一次矩阵计算

def softmax(Z):
    Z = np.exp(Z - Z.max(axis=-1, keepdims=True))
    return  Z / Z.sum(axis=-1, keepdims=True)

def self_attention(X, mask, W_KQV, W_out):
    K,Q,V = np.split(X@W_KQV, 3, axis=1)
    attn = softmax(K@Q.T/np.sqrt(X.shape[1])+mask)
    return attn@V@W_out, attn
    


In [3]:
T, d = 100, 64
attn = nn.MultiheadAttention(embed_dim=d, num_heads=1, bias=False, batch_first=True)
M = torch.triu(-float("inf")*torch.ones(T,T), diagonal=1)#对角线也为 0
X = torch.randn(1,T,d)
Y_, A_ = attn(X, X, X, attn_mask=M)

In [4]:
attn.out_proj.weight.shape

torch.Size([64, 64])

In [5]:
Y, A = self_attention(X[0].numpy()
                      , mask=M.numpy()
                      , W_KQV=attn.in_proj_weight.detach().numpy().T
                      , W_out=attn.out_proj.weight.detach().numpy().T)

In [6]:
np.linalg.norm(A-A_[0].detach().numpy())

np.float64(1.8337079558827791e-07)

## Minibatching

**下面的 res1 和 res2 的计算方式其实是等价的，而且没有任何 memory copy，属于普通的矩阵乘法**

In [7]:

B = np.random.randn(5,4, 10,3)
C = np.random.randn(3,6)
res1 = B@C



res2 = (B.reshape(-1,3)@C).reshape(res1.shape)

print(f"res1.shape:{res1.shape}, res2.shape:{res2.shape}")
np.linalg.norm(res1-res2)

res1.shape:(5, 4, 10, 6), res2.shape:(5, 4, 10, 6)


np.float64(0.0)

In [8]:
 # illustration of batch matmul
B = np.random.randn(10,3,5,4)
C = np.random.randn(10,3,4,3)
(B@C).shape

(10, 3, 5, 3)

In [9]:

def self_attention(X, mask, W_KQV, W_out):
    K,Q,V = np.split(X@W_KQV, 3, axis=-1) # axis为-1，兼容 batch和非 batch 你
    attn = softmax(K@Q.swapaxes(-1,-2)/np.sqrt(X.shape[-1])+mask)
    return attn@V@W_out, attn

In [10]:
B, T, d = 10, 100, 64
attn = nn.MultiheadAttention(embed_dim=d, num_heads=1, bias=False, batch_first=True)
M = torch.triu(-float("inf")*torch.ones(T,T), diagonal=1)#对角线也为 0
X = torch.randn(B,T,d)
Y_, A_ = attn(X, X, X, attn_mask=M)


Y, A = self_attention(X.numpy(), M.numpy(),
                      attn.in_proj_weight.detach().numpy().T, 
                      attn.out_proj.weight.detach().numpy().T)

In [11]:
print(np.linalg.norm(A - A_.detach().numpy()))
print(np.linalg.norm(Y - Y_.detach().numpy()))

5.928026000711085e-07
4.859055827691805e-06


## MultiHead Attentation

In [12]:
def multihead_attention(X, mask, heads, W_KQV, W_out):
    B,T,d = X.shape
    h = heads
    K,Q,V = np.split(X@W_KQV, 3, axis=-1)
    # B,T,d -> B,h,T,d/h
    K,Q,V = [a.reshape(B,T,h, d//h).swapaxes(1,2) for a in (K,Q,V)]
    
    attn = softmax(K@Q.swapaxes(-1,-2)/ np.sqrt(d//h)+mask)
    # B,h,T,d/h -> B,T,d
    return (attn@V).swapaxes(1,2).reshape(B,T,d)@W_out, attn

In [13]:
heads = 4
attn = nn.MultiheadAttention(d, heads, bias=False, batch_first=True)
Y_, A_ = attn(X,X,X, attn_mask=M)

In [14]:
Y, A = multihead_attention(X.numpy(), M.numpy(), heads,
                      attn.in_proj_weight.detach().numpy().T, 
                      attn.out_proj.weight.detach().numpy().T)
                           

In [15]:
print(np.linalg.norm(A.mean(axis=1) - A_.detach().numpy()))
print(np.linalg.norm(Y - Y_.detach().numpy()))

3.584518926386347e-07
4.915171476680839e-06


In [16]:
A.shape, A_.shape

((10, 4, 100, 100), torch.Size([10, 100, 100]))

## Transfomrer Block

In [17]:
def relu(Z):
    return np.maximum(Z,0)

def layer_norm(Z, eps):
    mean = np.mean(Z, keepdims=True, axis=-1)
    var = np.var(Z, keepdims=True, axis=-1)
    return (Z-mean)/np.sqrt(var+eps)

def transformer(X, mask, heads, W_KQV, W_out, W_ff1, W_ff2, eps):
    Y, attn = multihead_attention(X, mask, heads, W_KQV, W_out)
    Z = layer_norm(X+Y, eps)
    return layer_norm(Z+relu(Z@W_ff1)@W_ff2, eps)

In [18]:
trans = nn.TransformerEncoderLayer(d, heads, dim_feedforward=128, dropout=0.0, batch_first=True)
trans.linear1.bias.data.zero_()
trans.linear2.bias.data.zero_()
Y_ = trans(X, M)

In [19]:
Y = transformer(X.numpy(), M.numpy(), heads,
                trans.self_attn.in_proj_weight.detach().numpy().T,  # W_KQV
                trans.self_attn.out_proj.weight.detach().numpy().T,  # W_out
                trans.linear1.weight.detach().numpy().T,
                trans.linear2.weight.detach().numpy().T,
                trans.norm1.eps
               )

In [20]:
X.shape

torch.Size([10, 100, 64])

In [21]:
print(np.linalg.norm(Y - Y_.detach().numpy()))

2.3527511131683508e-05
